# 02 — Model Comparison: So Sánh Các LLM Cục Bộ

**Vai trò:** Model Engineer · **Task:** S4-ME-01 (Yêu cầu 9.4)

`config/models.yaml` liệt kê 3 LLM được hỗ trợ — `llama3`, `mistral`, `gemma` — mà người dùng có thể chọn ngay trên sidebar của dashboard (Yêu cầu 10.2). Notebook này benchmark các model **đã được pull về máy** với cùng một prompt, đo `latency_ms` qua `OllamaClient.generate()` (S3-ME-01) và so sánh độ dài câu trả lời — giúp người học chọn model phù hợp giữa đánh đổi tốc độ/chất lượng.

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.generation.llm_client import OllamaClient
from src.pipeline.experiment_tracker import ExperimentTracker

print(f"Project root: {PROJECT_ROOT}")

## 1. Phát hiện model nào đã pull về OLLAMA

`list_models()` không bao giờ ném exception (Yêu cầu 5.4) — ta dùng nó để lọc ra, trong số 3 model ứng viên (`config/models.yaml`), những model **thực sự khả dụng** trên máy. Nếu chưa pull đủ, notebook vẫn chạy hết — chỉ benchmark trên tập con khả dụng (đúng tinh thần "không có exception chưa xử lý dù thiếu model", DoD Sprint 4).

In [ ]:
probe_client = OllamaClient()
server_available = probe_client.is_available()
pulled_models = probe_client.list_models()

CANDIDATE_MODELS = ["llama3", "mistral", "gemma"]
usable_models = [
    name for name in CANDIDATE_MODELS
    if any(pulled.split(":")[0] == name for pulled in pulled_models)
]

print(f"OLLAMA server kha dung tai {probe_client.base_url}: {server_available}")
print(f"Model da pull             : {pulled_models or '(khong co)'}")
print(f"Model ung vien co the so sanh: {usable_models or '(chua co model nao trong config/models.yaml duoc pull)'}")

if not usable_models:
    print(
        "\n⚠️  Chua co model ung vien nao duoc pull. Hay chay:\n"
        "    ollama pull llama3\n"
        "    ollama pull mistral\n"
        "    ollama pull gemma\n"
        "roi chay lai notebook de benchmark thuc."
    )

## 2. Benchmark — cùng một prompt, đo `latency_ms` cho từng model

Mỗi model được gọi qua một `OllamaClient` riêng (cùng `temperature`/`max_tokens` để so sánh công bằng). `generate()` trả lỗi mô tả rõ URL khi server không khả dụng (Yêu cầu 5.5) — notebook bọc mỗi lần gọi trong `try/except` để một model lỗi không làm hỏng cả vòng lặp benchmark.

In [ ]:
PROMPT = "Giai thich ngan gon RAG (Retrieval-Augmented Generation) la gi, trong khoang 3 cau."

benchmark_results = []
for model_name in usable_models:
    client = OllamaClient(model_name=model_name, temperature=0.7, max_tokens=200)
    start = time.perf_counter()
    try:
        answer = client.generate(PROMPT)
        latency_ms = (time.perf_counter() - start) * 1000
        benchmark_results.append({
            "model": model_name,
            "latency_ms": latency_ms,
            "answer_length_chars": len(answer),
            "answer_preview": answer.strip().replace("\n", " ")[:160],
        })
        print(f"[{model_name}] {latency_ms:.0f} ms — {len(answer)} ky tu")
    except Exception as exc:
        print(f"[{model_name}] loi khi sinh cau tra loi: {exc}")

if not benchmark_results:
    print("\nKhong co ket qua benchmark thuc — xem huong dan o muc 1 de pull them model.")

## 3. Bảng so sánh kết quả

Tổng hợp `latency_ms` và độ dài câu trả lời của từng model vào một `DataFrame` để so sánh trực quan — model nào nhanh hơn, model nào trả lời dài/chi tiết hơn với cùng một câu hỏi.

In [ ]:
if benchmark_results:
    df_benchmark = pd.DataFrame(benchmark_results).sort_values("latency_ms")
    display(df_benchmark[["model", "latency_ms", "answer_length_chars", "answer_preview"]])
else:
    df_benchmark = pd.DataFrame(columns=["model", "latency_ms", "answer_length_chars", "answer_preview"])
    print("Bang trong — chua co model nao duoc benchmark (xem muc 1).")

## 4. Ghi lại thực nghiệm qua `ExperimentTracker`

Mỗi lượt benchmark được ghi như một sự kiện "query" (Yêu cầu 9.8, S4-PE-01) — `params["question"]` lưu cả tên model để phân biệt khi xem lại ở trang **Experiment Log** hoặc khi `compare_sessions()` giữa các phiên benchmark khác nhau.

In [ ]:
tracker = ExperimentTracker()
for row in benchmark_results:
    tracker.log_query(
        question=f"[model_comparison:{row['model']}] {PROMPT}",
        top_k=0,
        contexts=[],
        answer=row["answer_preview"],
        latency_ms=row["latency_ms"],
    )

print("Tom tat phien thuc nghiem:")
print(tracker.get_summary())

## 5. Tổng kết

- `OllamaClient.list_models()` cho biết model nào đã sẵn sàng để benchmark — notebook tự thích nghi với những gì đã được pull, không giả định cứng một model cụ thể nào (Yêu cầu 9.4).
- `latency_ms` đo bằng `time.perf_counter()` quanh `generate()` phản ánh tốc độ sinh câu trả lời thực tế của từng LLM cục bộ qua OLLAMA — đây chính là phép đo nền tảng mà `RAGPipeline.query()` dùng để báo cáo `RAGResponse.latency_ms` (Yêu cầu 7.4).
- Kết quả benchmark được ghi lại qua `ExperimentTracker.log_query()` mà không làm gián đoạn notebook (Yêu cầu 9.8) — có thể lưu lại bằng `save_session()` và đối chiếu giữa các lần thử với `compare_sessions()` ở trang Experiment Log của dashboard.